In [ ]:
# 1. Check the GPU: all team runs use the same GPU type (L4)
import torch
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, f"{gpu.total_memory / 2**30:.1f} GiB")
assert "L4" in gpu.name, "Switch the runtime to an L4 GPU (Runtime > Change runtime type)"

In [ ]:
# 2. Get a fresh copy of the branch with scripts/model_filter.py (switch to main once it is merged)
BRANCH = "model-filter"
%cd /content
!rm -rf kv-evidence-loss
!git clone -q -b {BRANCH} https://github.com/burakzor/kv-evidence-loss.git
%cd /content/kv-evidence-loss
!git log -1 --format="%h %ad %s" --date=short

In [ ]:
# 3. Install the pinned team environment
# kvpress 0.5.5 needs transformers<5.3 and pandas<3; torch is Colab's own CUDA build
!pip install -q -r requirements.txt
!pip install -q kvpress==0.5.5 transformers==5.2.0 pandas==2.2.3
!pip list 2>/dev/null | grep -iE "^(torch|transformers|tokenizers|kvpress|accelerate|datasets|pandas|numpy|pyarrow|tqdm) "

In [ ]:
# 4. Record the full environment of this run (GPU, driver, Python, CUDA, every package)
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"; pip freeze) > data/hotpotqa/filter_qwen3_8b_env.txt
!head -4 data/hotpotqa/filter_qwen3_8b_env.txt

In [ ]:
# 5. Optional: Hugging Face token for a faster download (only works in the Colab web UI)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"No HF token ({type(e).__name__}); downloading anonymously.")

In [ ]:
# 6. Test on the first 2 questions (first run downloads Qwen3-8B, ~16 GB); writes filter_qwen3_8b_limit2.parquet
!python scripts/model_filter.py --limit 2

In [ ]:
# 7. Full run: 169 questions x 3 conditions (A only, B only, no paragraph)
!python scripts/model_filter.py

In [ ]:
# 8. Look at the questions the filter removes, with the answer of every condition
import pandas as pd
pd.set_option("display.max_colwidth", 60)
df = pd.read_parquet("data/hotpotqa/filter_qwen3_8b.parquet")
gold = pd.read_parquet("results/answers/full_1.0.parquet")["gold"]
print("kept:", df["keep"].sum(), "/", len(df))
removed = df[df["full_contains"] & ~df["keep"]].join(gold)
removed[["gold", "full_answer", "a_only_answer", "b_only_answer", "no_ctx_answer", "a_only_contains", "b_only_contains", "no_ctx_contains"]]

In [ ]:
# 9. Download the filter table and the environment record
from google.colab import files
files.download("data/hotpotqa/filter_qwen3_8b.parquet")
files.download("data/hotpotqa/filter_qwen3_8b_env.txt")